# 02 — Идентификация и неопределенность

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import ensure_dir, set_seed, save_dataframe, save_json, get_repo_root
from src.systems import cross_coupled_uncontrolled
from src.simulation import simulate_batch, uncertain_dynamics
from src.plots import set_plot_style, save_figure, plot_phase_trajectories, plot_residual_histogram, plot_lyapunov_contours
from src.basis import get_basis
from src.identification import fit_identified_model, predict_vector_field, rmse
from src.uncertainty import compute_residuals, residual_norms, estimate_epsilon, compute_bounding_box, bounded_disturbance
from src.lyapunov import numerical_jacobian, solve_lyapunov, evaluate_lyapunov_grid
from src.control import candidate_gains, closed_loop_jacobian

ROOT = get_repo_root()
DATA_DIR = ROOT / 'data' / 'processed'
FIG_DIR = ROOT / 'results' / 'figures'
METRICS_DIR = ROOT / 'results' / 'metrics'
TABLES_DIR = ROOT / 'results' / 'tables'
for p in [DATA_DIR, FIG_DIR, METRICS_DIR, TABLES_DIR]:
    ensure_dir(p)


In [ ]:
set_plot_style()
df = pd.read_csv(DATA_DIR / 'cross_coupled_dataset.csv')
X = df[['x1','x2']].to_numpy()
Xdot = df[['xdot1','xdot2']].to_numpy()

basis_fn = get_basis('quadratic_with_constant')
res = fit_identified_model(X, Xdot, basis_fn=basis_fn, basis_name='quadratic_with_constant', method='least_squares')
Xdot_hat = predict_vector_field(X, basis_fn, res.coefficients)
residual = compute_residuals(Xdot, Xdot_hat)
norms = residual_norms(residual)
epsilon = estimate_epsilon(norms, q=0.95)

save_json({'rmse': rmse(Xdot, Xdot_hat), 'epsilon_q95': epsilon}, METRICS_DIR / 'identification_metrics.json')
lo, up = compute_bounding_box(X)
save_json({'omega_lower': lo.tolist(), 'omega_upper': up.tolist()}, METRICS_DIR / 'omega_box.json')

plt.figure(); plot_residual_histogram(norms, epsilon); save_figure(FIG_DIR / 'residual_hist.png'); plt.show()

def fhat(t, x):
    return predict_vector_field(x[None, :], basis_fn, res.coefficients)[0]

initials = np.array([[-1.2,-0.8],[-1.0,0.7],[-0.6,1.1],[0.5,-1.0],[1.0,0.9],[1.3,-0.4]])
t_eval = np.linspace(0,10,400)
traj_hat = simulate_batch(fhat, initials, (0,10), t_eval)
plt.figure(); plot_phase_trajectories(traj_hat, 'Identified model phase trajectories'); save_figure(FIG_DIR / 'phase_identified.png'); plt.show()
print('RMSE:', rmse(Xdot, Xdot_hat), 'epsilon:', epsilon)
